# 第 20 章 · 策略梯度与 REINFORCE：不学价值，直接调策略

本章回答：能不能跳过 Q，直接把策略本身调好？"让好动作更常出现"怎么写成数学？log 和那个"方便的恒等式"是干什么的？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/20-policy-gradient.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np
import torch

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| θ = [0, 0] 的策略 | `softmax(theta)` | [0.5, 0.5] |
| ∇log π(机器 1) | `grad_log_softmax(theta, 1)` | [−0.5, 0.5] |
| autograd 核对 | `torch.log_softmax(t, 0)[a].backward()` | [−0.5, 0.5] |
| REINFORCE 500 步 | `reinforce_bandit()` | 0.547、0.516、0.573 … 0.992 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `autograd_check`**（来自 `ch20_policy_gradient.py`）：下面要用到的小工具。

In [ ]:
def autograd_check(theta, a):
    t = torch.tensor(theta, requires_grad=True)
    torch.log_softmax(t, dim=0)[a].backward()
    return t.grad.numpy()

**函数 `softmax`**（来自 `ch20_policy_gradient.py`）：下面要用到的小工具。

In [ ]:
def softmax(theta):
    e = np.exp(theta - theta.max())
    return e / e.sum()

**函数 `grad_log_softmax`**（来自 `ch20_policy_gradient.py`）：∇_θ log π(a) = one_hot(a) − π（softmax 策略的现成结论，下面用 autograd 核对）。

In [ ]:
def grad_log_softmax(theta, a):
    """∇_θ log π(a) = one_hot(a) − π（softmax 策略的现成结论，下面用 autograd 核对）。"""
    g = -softmax(theta)
    g[a] += 1
    return g

**函数 `reinforce_bandit`**（来自 `ch20_policy_gradient.py`）：两台老虎机，策略 = softmax(θ)。一局只有一步，所以回报 G 就是这一步的奖励。

In [ ]:
def reinforce_bandit(steps=500, alpha=0.1, seed=0, means=(1.0, 2.0)):
    """两台老虎机，策略 = softmax(θ)。一局只有一步，所以回报 G 就是这一步的奖励。"""
    rng, theta, hist = np.random.default_rng(seed), np.zeros(len(means)), []
    for _ in range(steps):
        pi = softmax(theta)
        a = rng.choice(len(means), p=pi)
        G = rng.normal(means[a], 1.0)
        theta += alpha * G * grad_log_softmax(theta, a)
        hist.append(softmax(theta)[1])
    return theta, hist

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
theta = np.array([0.0, 0.0])

In [ ]:
print("θ = [0, 0]：π =", softmax(theta), "；∇log π(右) =", grad_log_softmax(theta, 1))

In [ ]:
print("autograd 核对：", autograd_check(theta, 1))

In [ ]:
th, hist = reinforce_bandit()

In [ ]:
print("REINFORCE 500 步后 π(好机器) =", round(hist[-1], 3), "；前几步", [round(h, 3) for h in hist[:3]])

## 3. 代码里才看得到的

真正的深度 RL 代码不手写梯度，而是写一个"假的 loss"让 PyTorch 求导：`loss = -(log_prob * G).mean()`，`loss.backward()` 得到的正好是 $-G\nabla\log\pi$。CS 285 HW2 的 `update` 函数就是这样写的；这个 loss 的数值本身没有意义，只有它的梯度有意义。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换步长：`reinforce_bandit(alpha=0.01 / 0.1 / 1.0)` 第 50 步时好机器的概率是 0.589、0.889、0.996，第 500 步是 0.885、0.992、0.997。步子大学得快，但一开始抽错几次也会被放大。
2. θ = [1, 0] 时 `softmax` → [0.731, 0.269]，`grad_log_softmax(θ, 1)` → [−0.731, 0.731]：概率越小的动作被抽中时，推得越猛。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 不学价值直接调策略：$\pi_\theta$ 用 softmax，目标 $J(\theta)$ 要变大；
> log 是 e 次方反过来，导数 1/x；$\nabla\pi = \pi\nabla\log\pi$ 把梯度变成期望；
> $\nabla J = \mathbb{E}[G\,\nabla\log\pi]$：好的推高，坏的压低；
> REINFORCE：抽一局，按回报推一把；等终局、方差大。